In [43]:
import pandas as pd
import sqlite3

In [44]:
account_data = pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/account.csv',sep=';')
loan = pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/loan.csv',sep=';')
trans_data = pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/trans.csv',sep=';')

/var/folders/34/1yg1tqgd1b96v45w9k29prnw0000gn/T/ipykernel_37408/175045100.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  trans_data = pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/trans.csv',sep=';')


In [45]:
disp=pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/disp.csv',sep=';')
client=pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/client.csv',sep=';')
district=pd.read_csv('/Users/utkuseyithanoglu/Desktop/1999 czech banking/fraud_detection/data/district.csv',sep=';')

In [46]:
conn = sqlite3.connect('bank_data.db')

In [47]:
account_data.to_sql('account_data', conn, if_exists='replace', index=False)
loan.to_sql('loan', conn, if_exists='replace', index=False)
trans_data.to_sql('trans_data', conn, if_exists='replace', index=False)
disp.to_sql('disp', conn, if_exists='replace', index=False)
client.to_sql('client', conn, if_exists='replace', index=False)
district.to_sql('district', conn, if_exists='replace', index=False)

77

In [48]:
query =""" SELECT * 
FROM account_data 
LIMIT 5;"""
pd.read_sql(query, conn)

,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101
3,2378,16,POPLATEK MESICNE,930101
4,2632,24,POPLATEK MESICNE,930102


In [49]:
query =""" SELECT * 
FROM trans_data 
LIMIT 5;"""
pd.read_sql(query, conn)

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,930101,PRIJEM,VKLAD,700.0,700.0,None,None,None
1,171812,576,930101,PRIJEM,VKLAD,900.0,900.0,None,None,None
2,207264,704,930101,PRIJEM,VKLAD,1000.0,1000.0,None,None,None
3,1117247,3818,930101,PRIJEM,VKLAD,600.0,600.0,None,None,None
4,579373,1972,930102,PRIJEM,VKLAD,400.0,400.0,None,None,None


In [50]:
query = query = """
SELECT 
    loan.loan_id,
    loan.account_id,
    loan.amount,
    loan.duration,
    loan.payments,
    loan.status,
    account_data.frequency,
    disp.disp_id,
    disp.client_id,
    client.birth_number,
    client.district_id
FROM loan
JOIN account_data ON loan.account_id = account_data.account_id
JOIN disp ON account_data.account_id = disp.account_id
JOIN client ON disp.client_id = client.client_id
LIMIT 5;
"""

pd.read_sql(query, conn)


,loan_id,account_id,amount,duration,payments,status,frequency,disp_id,client_id,birth_number,district_id
0,5314,1787,96396,12,8033.0,B,POPLATEK TYDNE,2166,2166,475722,30
1,5316,1801,165960,36,4610.0,A,POPLATEK MESICNE,2181,2181,680722,46
2,6863,9188,127080,60,2118.0,A,POPLATEK MESICNE,11006,11314,360602,45
3,5325,1843,105804,36,2939.0,A,POPLATEK MESICNE,2235,2235,405420,14
4,7240,11013,274740,60,4579.0,A,POPLATEK TYDNE,13231,13539,780907,63


In [51]:
query = """SELECT loan.loan_id,
    loan.account_id,
    loan.amount,
    loan.duration,
    loan.payments,
    loan.status,
    account_data.frequency,
    disp.disp_id,
    disp.client_id,
    client.birth_number,
    client.district_id,
    COUNT(trans_data.account_id) AS total_trans,
    SUM(trans_data.amount) AS total_trans_amount
FROM loan
JOIN account_data ON loan.account_id = account_data.account_id
JOIN disp ON account_data.account_id = disp.account_id
JOIN client ON disp.client_id = client.client_id
LEFT JOIN trans_data ON account_data.account_id = trans_data.account_id
GROUP BY
    loan.loan_id,
    loan.account_id,
    loan.amount,
    loan.duration,
    loan.payments,
    loan.status,
    account_data.frequency,
    disp.disp_id,
    disp.client_id,
    client.birth_number,
    client.district_id
LIMIT 5;
"""

pd.read_sql(query, conn)

,loan_id,account_id,amount,duration,payments,status,frequency,disp_id,client_id,birth_number,district_id,total_trans,total_trans_amount
0,4959,2,80952,24,3373.0,A,POPLATEK MESICNE,2,2,450204,1,478,3151479.3
1,4959,2,80952,24,3373.0,A,POPLATEK MESICNE,3,3,406009,1,478,3151479.3
2,4961,19,30276,12,2523.0,B,POPLATEK MESICNE,25,25,395423,21,303,1575515.9
3,4962,25,30276,12,2523.0,A,POPLATEK MESICNE,31,31,620209,68,274,2958545.1
4,4967,37,318480,60,5308.0,D,POPLATEK MESICNE,45,45,520826,20,130,948153.9


In [52]:
query = """SELECT loan.loan_id,
    loan.account_id,
    loan.amount,
    loan.duration,
    loan.payments,
    loan.status,
    account_data.frequency,
    disp.disp_id,
    disp.client_id,
    client.birth_number,
    client.district_id,
    COUNT(trans_data.account_id) AS total_trans,
    SUM(trans_data.amount) AS total_trans_amount
FROM loan
JOIN account_data ON loan.account_id = account_data.account_id
JOIN disp ON account_data.account_id = disp.account_id
JOIN client ON disp.client_id = client.client_id
LEFT JOIN trans_data ON account_data.account_id = trans_data.account_id
GROUP BY
    loan.loan_id,
    loan.account_id,
    loan.amount,
    loan.duration,
    loan.payments,
    loan.status,
    account_data.frequency,
    disp.disp_id,
    disp.client_id,
    client.birth_number,
    client.district_id;
"""

pd.read_sql(query, conn)

,loan_id,account_id,amount,duration,payments,status,frequency,disp_id,client_id,birth_number,district_id,total_trans,total_trans_amount
0,4959,2,80952,24,3373.0,A,POPLATEK MESICNE,2,2,450204,1,478,3151479.3
1,4959,2,80952,24,3373.0,A,POPLATEK MESICNE,3,3,406009,1,478,3151479.3
2,4961,19,30276,12,2523.0,B,POPLATEK MESICNE,25,25,395423,21,303,1575515.9
3,4962,25,30276,12,2523.0,A,POPLATEK MESICNE,31,31,620209,68,274,2958545.1
4,4967,37,318480,60,5308.0,D,POPLATEK MESICNE,45,45,520826,20,130,948153.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
822,7295,11328,280440,60,4674.0,C,POPLATEK MESICNE,13616,13924,525909,54,146,1326820.0
823,7304,11349,419880,60,6998.0,C,POPLATEK TYDNE,13647,13955,456030,1,304,3957372.2
824,7304,11349,419880,60,6998.0,C,POPLATEK TYDNE,13648,13956,430406,1,304,3957372.2
825,7305,11359,54024,12,4502.0,A,POPLATEK MESICNE,13660,13968,680413,61,378,2948081.4


In [53]:
main_df = pd.read_sql(query, conn)

main_df.to_csv("final_loan_data.csv", index=False)

In [24]:
query="""SELECT loan.loan_id,loan.account_id,COUNT(*) as loan_count
FROM loan
GROUP BY loan.loan_id, loan.account_id
HAVING COUNT(*) > 1;"""
pd.read_sql(query, conn)

,loan_id,account_id,loan_count


In [26]:
query = """SELECT 
    status,
    COUNT(*) AS total_loans
FROM loan
GROUP BY status;"""
pd.read_sql(query, conn)

,status,total_loans
0,A,203
1,B,31
2,C,403
3,D,45


In [29]:
query = """SELECT AVG(amount) AS avg_loan_amount
FROM loan;"""
pd.read_sql(query, conn)

,avg_loan_amount
0,151410.175953


In [33]:
query = """
SELECT 
    loan.status,
    AVG(trans_data.amount) AS avg_trans_amount
FROM loan
LEFT JOIN trans_data 
    ON loan.account_id = trans_data.account_id
GROUP BY loan.status;
"""

pd.read_sql(query, conn)

,status,avg_trans_amount
0,A,8290.889501
1,B,8454.100898
2,C,8238.054805
3,D,8303.196085
